# 1. Baseline : NLLB-200 (zero-shot) FR <-> Ewe

**Objectif** : mesurer la qualite de traduction du modele **NLLB-200-distilled-600M**
(Meta AI) sur notre corpus de test **sans aucun entrainement** (mode "zero-shot").

C'est la **reference de depart** : tout le travail de fine-tuning (notebook 2)
devra faire mieux que ces scores.

## Comment ca marche ?

- **NLLB** ("No Language Left Behind") est un modele de traduction multilingue
  entraine sur 200 langues, dont l'**ewe** (code `ewe_Latn`).
- Il est **"zero-shot"** pour nous : il n'a jamais vu notre corpus, mais il a vu
  de l'ewe pendant son entrainement.
- On mesure la qualite avec deux metriques standard :
  - **chrF++** (la metrique principale du projet, robuste aux petites variations)
  - **BLEU** (metrique classique, plus stricte)

> Le test set est charge depuis le **repo GitHub public** du projet.
> C'est le split `test.tsv` : 6 564 paires jamais utilisees pour l'entrainement.

In [1]:
# Installation des bibliotheques necessaires
# - transformers : modeles HuggingFace (NLLB)
# - sacrebleu    : metriques chrF++ et BLEU
# - pandas       : lecture des fichiers TSV
# - sentencepiece : tokenizer de NLLB (obligatoire)
!pip install -q transformers sacrebleu pandas sentencepiece datasets

print("Dependances installees")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.3 MB/s eta 0:00:00
Dependances installees


In [2]:
# Diagnostic GPU
# Colab fournit un GPU (T4) gratuitement, mais il faut l'activer :
#   menu Executer > Changer le type d'execution > T4 GPU
#   puis Executer > Redemarrer la session (obligatoire).
import torch

print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("Memoire :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "Go")
else:
    print("Attention : execution sur CPU (lent). Active le GPU T4 puis redemarre la session.")
    print("Si Colab ne propose pas de GPU (quota), utilise Kaggle : Accelerator > GPU T4.")

CUDA disponible : True
GPU : Tesla T4
Memoire : 15.6 Go


In [3]:
# Imports
import torch
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilise :", device)

Device utilise : cuda


In [5]:
# Chargement du jeu de test depuis le repo GitHub public
URL_TEST = "https://raw.githubusercontent.com/cherif-tg/tg_nlp_toolkit/main/data/processed/v0.3/test.tsv"
from google.colab import files; upload = files.upload()
try:
    df = pd.read_csv(test.csv, sep="\t",on_bad_lines="skip")
    print("Test set charge :", len(df), "paires FR<->Ewe")
    print(df.head(3))
except Exception as e:
    print("Telechargement GitHub impossible :", e)
    print("Solution : telecharge test.tsv depuis le repo et execute :")
    print("  from google.colab import files; upload = files.upload()")

Saving test.tsv to test.tsv
Telechargement GitHub impossible : name 'test' is not defined
Solution : telecharge test.tsv depuis le repo et execute :
  from google.colab import files; upload = files.upload()


In [9]:
df = pd.read_csv("test.tsv", sep="\t",on_bad_lines="skip")

df.head(3)

,source,fr,ewe
0,bible,"Ils ont regardé, tout stupéfaits,","Esi wokpoe la, wofe nu ku, dzidzi fo wo, eye wosi"
1,nllb,"C'est une drôle de question, non ?",Ðe biabia sia mele vevie ŋutɔ oa?
2,nllb,Les Juifs enterraient leurs morts tout de suit...,"Yudatɔwo ɖia woƒe ame kukuwo kaba, zi geɖe le ..."


In [12]:
# Chargement du modele NLLB-200-distilled-600M
# 600M parametres = version "distilled" (legere), adaptee a un GPU gratuit.
MODEL_NAME = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# Verification des codes de langue
assert tokenizer.convert_tokens_to_ids("fra_Latn") != tokenizer.unk_token_id, "francais absent"
assert tokenizer.convert_tokens_to_ids("ewe_Latn") != tokenizer.unk_token_id, "ewe absent"
print("Modele charge - codes langue : fra_Latn (fr), ewe_Latn (ewe)")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Modele charge - codes langue : fra_Latn (fr), ewe_Latn (ewe)


In [13]:
# Fonction de traduction en batch
# - src / tgt : codes de langue NLLB (fra_Latn, ewe_Latn)
# - num_beams=4 : recherche en faisceau (meilleure qualite que greedy)
# - Le tokenizer doit connaitre la langue SOURCE avant d'encoder.

def traduire(textes, src="fra_Latn", tgt="ewe_Latn", max_len=128, batch_size=16):
    tokenizer.src_lang = src
    resultats = []
    for i in range(0, len(textes), batch_size):
        lot = textes[i:i + batch_size]
        enc = tokenizer(lot, return_tensors="pt", padding=True,
                        truncation=True).to(device) # Removed max_length here
        with torch.no_grad():
            gen = model.generate(
                **enc,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
                max_new_tokens=max_len, # max_new_tokens is now the sole length control
                num_beams=4,
            )
        resultats += tokenizer.batch_decode(gen, skip_special_tokens=True)
    return resultats

print("Fonction de traduction prete")

Fonction de traduction prete


In [14]:
# Evaluation FR -> EWE (le sens qui nous interesse le plus)
# On traduit les 6 564 phrases francaises du test set, puis on compare
# aux traductions ewe de reference avec chrF++ et BLEU.

preds_fr_ee = traduire(df["fr"].tolist(), src="fra_Latn", tgt="ewe_Latn")
refs_ee = df["ewe"].tolist()



[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

AttributeError: module 'sacrebleu' has no attribute 'corpus'

In [17]:
from sacrebleu.metrics import CHRF, BLEU

# Instantiate the metric calculators
chrf_metric = CHRF()
bleu_metric = BLEU()

# Ensure all predictions and references are strings
preds_fr_ee_str = [str(x) for x in preds_fr_ee]
refs_ee_str = [str(x) for x in refs_ee]

# Calculate scores
chrf_fr_ee = chrf_metric.corpus_score(preds_fr_ee_str, [refs_ee_str])
bleu_fr_ee = bleu_metric.corpus_score(preds_fr_ee_str, [refs_ee_str])

print("FR -> EWE (zero-shot)")
print("   chrF++ :", round(chrf_fr_ee.score, 2))
print("   BLEU   :", round(bleu_fr_ee.score, 2))

# Afficher 3 exemples concrets
for i in range(3):
    print("--- Exemple", i + 1, "---")
    print("FR :", df['fr'].iloc[i])
    print("Ref:", refs_ee[i])
    print("Pred:", preds_fr_ee[i])

FR -> EWE (zero-shot)
   chrF++ : 34.96
   BLEU   : 11.38
--- Exemple 1 ---
FR : Ils ont regardé, tout stupéfaits,
Ref: Esi wokpoe la, wofe nu ku, dzidzi fo wo, eye wosi
Pred: Ame siwo katã nɔ afi ma la ƒe mo wɔ yaa eye woƒe mo wɔ yaa.
--- Exemple 2 ---
FR : C'est une drôle de question, non ?
Ref: Ðe biabia sia mele vevie ŋutɔ oa?
Pred: Nyabiase ɖedzesi aɖee wònye, alo ɖe?
--- Exemple 3 ---
FR : Les Juifs enterraient leurs morts tout de suite après leur décès, en général dans la journée même.
Ref: Yudatɔwo ɖia woƒe ame kukuwo kaba, zi geɖe le ŋkeke si dzi amea ku le.
Pred: Yudatɔwo nɔa woƒe ame kukuwo ɖi ge le woƒe ku megbe enumake, zi geɖe le ŋkeke ma ke dzi.


In [19]:
# Evaluation EWE -> FR (sens inverse)
# Ensure input texts are strings before passing to the translation function
ewe_texts_for_translation = [str(x) for x in df["ewe"].tolist()]
preds_ee_fr = traduire(ewe_texts_for_translation, src="ewe_Latn", tgt="fra_Latn")

# Ensure reference texts are strings for sacrebleu
refs_fr = [str(x) for x in df["fr"].tolist()]

# Calculate scores
chrf_ee_fr = chrf_metric.corpus_score(preds_ee_fr, [refs_fr])
bleu_ee_fr = bleu_metric.corpus_score(preds_ee_fr, [refs_fr])


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

--- Exemple 1 ---
FR : Ils ont regardé, tout stupéfaits,
Ref: Ils ont regardé, tout stupéfaits,
Pred: Mais ils les repoussèrent, et ils se mirent à cracher, à crier, à crier, et à s'enfuir.
--- Exemple 2 ---
FR : C'est une drôle de question, non ?
Ref: C'est une drôle de question, non ?
Pred: Cette question n'est- elle pas très importante?
--- Exemple 3 ---
FR : Les Juifs enterraient leurs morts tout de suite après leur décès, en général dans la journée même.
Ref: Les Juifs enterraient leurs morts tout de suite après leur décès, en général dans la journée même.
Pred: Et les Juifs ensevelirent les morts selon la coutume d'un jour.
EWE -> FR (zero-shot)
   chrF++ : 33.76
   BLEU   : 13.53
Tableau de bord baseline :


NameError: name 'blef_fr_ee' is not defined

In [21]:

# Afficher 3 exemples concrets
for i in range(3):
    print("--- Exemple", i + 1, "---")
    print("EWE :", df['ewe'].iloc[i])
    print("Ref:", refs_fr[i])
    print("Pred:", preds_ee_fr[i])

print("EWE -> FR (zero-shot)")
print("   chrF++ :", round(chrf_ee_fr.score, 2))
print("   BLEU   :", round(bleu_ee_fr.score, 2))

print("Tableau de bord baseline :")
print("   FR->EWE : chrF++", round(chrf_fr_ee.score, 2), "| BLEU", round(bleu_fr_ee.score, 2))
print("   EWE->FR : chrF++", round(chrf_ee_fr.score, 2), "| BLEU", round(bleu_ee_fr.score, 2))

--- Exemple 1 ---
EWE : Esi wokpoe la, wofe nu ku, dzidzi fo wo, eye wosi
Ref: Ils ont regardé, tout stupéfaits,
Pred: Mais ils les repoussèrent, et ils se mirent à cracher, à crier, à crier, et à s'enfuir.
--- Exemple 2 ---
EWE : Ðe biabia sia mele vevie ŋutɔ oa?
Ref: C'est une drôle de question, non ?
Pred: Cette question n'est- elle pas très importante?
--- Exemple 3 ---
EWE : Yudatɔwo ɖia woƒe ame kukuwo kaba, zi geɖe le ŋkeke si dzi amea ku le.
Ref: Les Juifs enterraient leurs morts tout de suite après leur décès, en général dans la journée même.
Pred: Et les Juifs ensevelirent les morts selon la coutume d'un jour.
EWE -> FR (zero-shot)
   chrF++ : 33.76
   BLEU   : 13.53
Tableau de bord baseline :
   FR->EWE : chrF++ 34.96 | BLEU 11.38
   EWE->FR : chrF++ 33.76 | BLEU 13.53


## Comment interpreter ces scores ?

- **chrF++ 40-55** sur cette tache : le modele "se debrouille" (le vocabulaire
  religieux est bien connu de NLLB).
- **BLEU bas (< 15)** : normal, BLEU est tres strict sur les mots exacts, et
  l'ewe de 1913 a une orthographe differente de l'ewe moderne vu par NLLB.
- Ces scores sont notre **reference** : le notebook 2 (fine-tuning LoRA sur
  notre corpus) doit les **depasser**, surtout en chrF++.

> Si le score est tres bas, verifie que le GPU est actif
> (menu Executer > Changer le type d'execution > T4 GPU).